In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

from sklearn.preprocessing import MultiLabelBinarizer
# used when each example can have multiple labels, as is true in this case 
# this converts a classifier into a vector. think action = 1, fantasy = 1, comedy = 0

from sklearn.feature_extraction.text import TfidfVectorizer
# turns text into numbers using TF-IDF
# vectorizer = TfidfVectorizer(stop_words = 'english', ngram_range = (1, 2))

from sklearn.multiclass import OneVsRestClassifier
# onevsrestclassifier is used for multilabel classification 
# rather than one giant model, onevsrest classifier trains one classifier for action vs not action, etc. 
# so, builds classifiers for each genre in this case. (we will probably have to fix this)

from sklearn.linear_model import LogisticRegression
# the actual learning algorithm. good thing i know logistic regression 

from sklearn.metrics import classification_report, f1_score
# for the evaluation metrics 

In [3]:
# import data
anime_df = pd.read_csv("anime_dataset.csv")
manga_df = pd.read_csv("manga_dataset.csv")
df = pd.concat([anime_df, manga_df])

In [4]:
# cleaning steps 
# only really care about title_english, synopsis, genres 

df = df.dropna(subset = ['synopsis', 'genres'])
df = df[['title', 'title_english', 'synopsis', 'genres']] # need double brackets to subset a dataframe

# we use title as a fallback in case title_english doesn't exist 
df['title_english'] = df['title_english'].fillna(df['title'])
df = df[['title_english', 'synopsis', 'genres']] 
df

,title_english,synopsis,genres
0,Cowboy Bebop,"Crime is timeless. By the year 2071, humanity ...",Action|Award Winning|Sci-Fi
1,Cowboy Bebop: The Movie,"Another day, another bounty—such is the life o...",Action|Sci-Fi
2,Trigun,"Vash the Stampede is the man with a $$60,000,0...",Action|Adventure|Sci-Fi
3,Witch Hunter Robin,"Though hidden away from the general public, Wi...",Action|Drama|Mystery|Supernatural
4,Beet the Vandel Buster,It is the dark century and the people are suff...,Action|Adventure|Fantasy
...,...,...,...
82884,Assistant Manager Kim Hates K-Pop Idols,Assistant Manager Kim Iwol has spent five year...,Drama
82885,Elf-Like Hana's Casual Apartment Life,"Hana, who lives in the cheap apartment next do...",Comedy|Supernatural
82886,Garden in the Closet,Kouki is a high school student with a large bu...,Boys Love|Drama|Supernatural
82887,Anta ga Warui,"""I am only interested in people who seem into ...",Drama|Girls Love


In [5]:
df['text'] = (df['title_english'] + " " + df['synopsis']) 

X_text = df['text'] # predictor variables

df["genre_list"] = df["genres"].apply(lambda x: x.split("|"))

In [6]:
df.to_csv("genre_list_included.csv") # so i can look at it. seems good 

In [7]:
# multi label binary coding is used when each example can have multiple labels, as is true in this case 
# this converts a classifier into a vector. think action = 1, fantasy = 1, comedy = 0
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['genre_list'])

In [8]:
# train/test split
df = df.reset_index(drop=True)
# X_train, X_test, y_train, y_test = train_test_split(X_text, Y, test_size = 0.2, random_state = 1001)

# In an effort to avoid index issues
train_df, test_df = train_test_split(df, test_size=0.2, random_state=1)


In [9]:
# TF-IDF vectorization 
vectorizer = TfidfVectorizer(max_features = 5000, stop_words = "english", ngram_range=(1, 2))

# fit_transform ONLY on training data
X_train_tfidf = vectorizer.fit_transform(X_train)

# transform ONLY on test data
X_test_tfidf = vectorizer.transform(X_test)

NameError: name 'X_train' is not defined

In [ ]:
# training model 
model = OneVsRestClassifier(LogisticRegression(max_iter = 5000))
model.fit(X_train_tfidf, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...max_iter=5000)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [ ]:
# making predictions 
y_pred = model.predict(X_test_tfidf)

In [ ]:
# evaluate !!!!
micro_f1 = f1_score(y_test, y_pred, average="micro")
macro_f1 = f1_score(y_test, y_pred, average="macro")

####### evaluation metrics for multi-class classification 
# precision: how many predicted yes were actually yes. minimizing false positives 
# recall: measures the ability to find all actual positive cases, minimizing false negatives 
# harmonic mean: H = n / (sum from 1 to n of 1 / x_i)
# F1: a machine learning metric that acts as the harmonic mean of precision and recall 
# ranges from 0 to 1
# macro-f1 calculates F1-score for each class independently and averages them 
# micro-f1 aggregates total true positives, false negatives, false positives across all classes

print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1: {macro_f1:.4f}") 

Micro F1: 0.4770
Macro F1: 0.3383


In [ ]:
# new_text = """A high school student discovers magical powers and must defend humanity from ancient demons."""

# new_text = """My Secret Depressed Husband A businessman decides to marry his secretary in order to
# keep good appearances, but they soon fall in love and grapple with 
# the awkwardness of revealing their true feelings. The secretary's dad dies.
# they live in a waste of a dystopia. the secretary has depression but the 
# businessman fights for her anyway"""

new_text = '''Super Stinky Trash Collectors Three friends in middle school decide to 
start collecting and recycling soda cans in order to raise money for their upcoming 
road trip, where they will make memories for a lifetime. '''

# Avant Garde|Award Winning|Drama|Sci-Fi|Suspense
# new_text = '''Fifteen years after a cataclysmic event known as the Second Impact, the world 
# faces a new threat: monstrous celestial beings called Angels invade Tokyo-3 one by one. 
# Mankind is unable to defend themselves against the Angels despite utilizing their most 
# advanced munitions and military tactics. The only hope for human salvation rests in the hands 
# of NERV, a mysterious organization led by the cold Gendou Ikari. NERV operates giant humanoid 
# robots dubbed "Evangelions" to combat the Angels with state-of-the-art advanced weaponry and 
# protective barriers known as Absolute Terror Fields. Years after being abandoned by his 
# father, Shinji Ikari, Gendou's 14-year-old son, returns to Tokyo-3. Shinji undergoes a 
# perpetual internal battle against the deeply buried trauma caused by the loss of his mother 
# and the emotional neglect he suffered at the hands of his father. Terrified to open himself 
# up to another, Shinji's life is forever changed upon meeting 29-year-old Misato Katsuragi, 
# a high-ranking '''

# new_text = '''An 18-year-old girl devotes herself to her religion and takes leave from school 
# in order to travel to South Korea for 18 months. She has many trials but ultimately learns
# the power of friendship and family, as well as the importance of God.'''

new_vector = vectorizer.transform([new_text])

# prediction = model.predict(new_vector) # default cutoff of 0.5
# predicted_genres = mlb.inverse_transform(prediction)
# print(predicted_genres)
# since this returned empty, that means model.predict(new_vector) returned all zeros 
# and the inverse transform returned empty, meaning no genres were predicted. 
# usually, the issue is that the default threshold is too strict. 

probabilities = model.predict_proba(new_vector)
for genre, prob in zip(mlb.classes_, probabilities[0]):
    if prob > 0.2:
        print(f"{genre}: {prob:.3f}")


NameError: name 'vectorizer' is not defined

In [ ]:
threshold = 0.3 # for the longer synopsis honestly a 0.5, 0.6 might be fine 
prediction = (probabilities >= threshold).astype(int)
predicted_genres = mlb.inverse_transform(prediction)
print(predicted_genres)

[('Action', 'Drama', 'Fantasy', 'Sci-Fi')]


In [ ]:
# type(X_test) # this is a pandas series

# # I want a dataframe with the title of the anime, the text, the true genre, and the predicted genre 
# # where are the predicted genres stored? 

# predicted_genres_matrix = mlb.inverse_transform(y_pred)
# print(predicted_genres_matrix)

# # now i need these predicted genres in a dataframe...? won't worry about formatting correctly for now 

# predicted_genres_matrix[4]

# type(predicted_genres_matrix) # this is a list 

# predicted_df = 

print(df.index[:10])
print(X_test.index[:10])

RangeIndex(start=0, stop=10, step=1)
Index([4612, 24382, 5105, 5422, 82512, 29862, 63204, 23281, 54020, 14384], dtype='int64')


In [ ]:
predicted_genres = mlb.inverse_transform(y_pred)
pred_df = pd.DataFrame({"predicted_genres": predicted_genres}, index = X_test.index)

print(pred_df.head())

actual_genres = mlb.inverse_transform(y_test)

results_df = pd.DataFrame({"actual_genres": actual_genres,"predicted_genres": predicted_genres}, index=X_test.index)

# print(results_df.head())

results_df = test_df[["title_english", "genres"]].copy()
results_df["predicted_genres"] = predicted_genres
results_df["predicted_genres"] = predicted_genres
results_df

           predicted_genres
4612                     ()
24382                    ()
5105           (Adventure,)
5422             (Romance,)
82512  (Boys Love, Erotica)


,title_english,genres,predicted_genres
24779,The young person's pure love,Boys Love|Drama|Erotica,()
33305,Karen with Zippers,Fantasy,()
55857,Shiawase Taputapu Sour,Hentai,"(Adventure,)"
25658,Angelique,Fantasy|Romance,"(Romance,)"
60880,Tatasen: Caramel-sensei no Kimyou na Kagai Jugyou,Comedy|Ecchi,"(Boys Love, Erotica)"
...,...,...,...
73928,Lens no Mukou no Megami-tachi,Comedy|Ecchi,"(Comedy,)"
36105,Tsuma ni Naru Hito,Girls Love,"(Action,)"
44975,Pavlov's Dog,Comedy,()
12977,Korasho no Kaitei Wakuwaku Daibouken! Movie,Adventure|Fantasy,()
